# Computational Theory Problems

## Problem 1: Binary Words and Operations

### Parity - x $\oplus$ y $\oplus$ z 
----
Used in rounds 20-39 & 64-79 of SHA-256 to operate on three 32-bit words to produce a 32-bit word output.
The output is the bitwise XOR of the three 32 bit "word" inputs.
Bitwise XOR meaning that each bit in the output is 1 if an odd number of the corresponding bits in the inputs are 1, and 0 otherwise.

Due to having 3 inputs, the parity function effectively counts the number of 1s in each bit position across the three inputs and sets the corresponding output bit to 1 if that count is odd, and to 0 if it is even.

e.g If we did a parity function that takes 2 bit inputs:<br>

> parity(00, 00, 01) would yield<br>
>- First bit: 0 + 0 + 0 = 0 (even) -> output 0<br>
>- Second bit: 0 + 0 + 1 = 1 (odd)  -> output 1<br>
>Resulting in output: 01<br>


In [6]:
import numpy as np
def parity(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word representing the parity (XOR) of x, y, z

    Performs bitwise XOR on three 32-bit words.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32(x ^ y ^ z)

def parityExamples():
    """
    Example cases demonstrating parity(x, y, z) == x ^ y ^ z (32-bit)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("single_bit",      np.uint32(0x00000001), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("two_same_bits",   np.uint32(0x00000001), np.uint32(0x00000001), np.uint32(0x00000000)),  # x ^ x == 0
        ("three_same_bits", np.uint32(0x00000001), np.uint32(0x00000001), np.uint32(0x00000001)),  # odd -> 1
        ("pattern_AAA",     np.uint32(0xAAAAAAAA), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("pattern_mix",     np.uint32(0x12345678), np.uint32(0x9ABCDEF0), np.uint32(0x0F0F0F0F)),
    ]

    print("Parity function examples.")
    # Test parity function against expected XOR results 
    for name, a, b, c in examples:
        res = np.uint32(parity(a, b, c))
        expected = np.uint32(a ^ b ^ c)
        assert res == expected, f"parity mismatch for {name}"
        print(f"{name}: a=0x{int(a):08x} b=0x{int(b):08x} c=0x{int(c):08x} -> parity=0x{int(res):08x} {int(res):032b}")
    print()

parityExamples()

Parity function examples.
all_zero: a=0x00000000 b=0x00000000 c=0x00000000 -> parity=0x00000000 00000000000000000000000000000000
single_bit: a=0x00000001 b=0x00000000 c=0x00000000 -> parity=0x00000001 00000000000000000000000000000001
two_same_bits: a=0x00000001 b=0x00000001 c=0x00000000 -> parity=0x00000000 00000000000000000000000000000000
three_same_bits: a=0x00000001 b=0x00000001 c=0x00000001 -> parity=0x00000001 00000000000000000000000000000001
pattern_AAA: a=0xaaaaaaaa b=0x00000000 c=0x00000000 -> parity=0xaaaaaaaa 10101010101010101010101010101010
pattern_mix: a=0x12345678 b=0x9abcdef0 c=0x0f0f0f0f -> parity=0x87878787 10000111100001111000011110000111



### Choose - (x $\land$ y)$\oplus$ ($\neg$ x $\land$ z)
----

This function uses x as a mask to select bits from y where x is 1, and from z where x is 0.
Used in rounds 0-19 of SHA-256.<br>
By performing bitwise operations, it effectively implements:<br>
>result = (x AND y) XOR (NOT x AND z)

e.g For a "2-bit choose"<br>
>ch(01, 10, 11) = 10<br>
>x = 01<br> 
>y = 1**0** x1 = 0 -> z1 = 1 <br>
>z = **1**1 x2 = 1 -> y2 = 0 <br>
>r = **10**

>|       | Bit 1 | Bit 2 |
>| ----- |:-----:| -----:|
>|   X   |  0    |   1   |
>|   Y   |  1    | **0** |
>|   Z   | **1** |   1   |
>|   r   | **1** | **0** |


In [10]:
def ch(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word 

    Chooses bits from y and z based on x.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32((x & y) | (np.uint32(~x) & z))

def chooseExamples():
    """
        Example cases demonstrating ch(x, y, z)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("x_all_ones",      np.uint32(0xFFFFFFFF), np.uint32(0x12345678), np.uint32(0x9ABCDEF0)),
        ("x_all_zeros",     np.uint32(0x00000000), np.uint32(0x12345678), np.uint32(0x9ABCDEF0)),
        ("mixed_bits",      np.uint32(0xF0F0F0F0), np.uint32(0xAAAAAAAA), np.uint32(0x55555555)),
    ]

    print("Choose function examples.")
    # Test ch function against expected results
    for name, a, b, c in examples:
        res = np.uint32(ch(a, b, c))
        expected = np.uint32((a & b) | (~a & c))
        assert res == expected, f"ch mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} y=0x{int(b):08x} z=0x{int(c):08x} -> ch=0x{int(res):08x} {int(res):032b}")
    print()


chooseExamples()

Choose function examples.
all_zero: x=0x00000000 y=0x00000000 z=0x00000000 -> ch=0x00000000 00000000000000000000000000000000
x_all_ones: x=0xffffffff y=0x12345678 z=0x9abcdef0 -> ch=0x12345678 00010010001101000101011001111000
x_all_zeros: x=0x00000000 y=0x12345678 z=0x9abcdef0 -> ch=0x9abcdef0 10011010101111001101111011110000
mixed_bits: x=0xf0f0f0f0 y=0xaaaaaaaa z=0x55555555 -> ch=0xa5a5a5a5 10100101101001011010010110100101



### Majority - (x $\land$ y) $\oplus$ (x $\land$ z) $\oplus$ (y $\land$ z)
----

Majority function: for each bit position, takes the majority value among x, y, z.

E.g For a "2-bit majority"<br>
>maj(01, 10, 11) = 11<br>
>x = 01<br> 
>y = 1**0** x1 = 0 -> z1 = 1 <br>
>y = 10<br> 
>x = 0**1** z0 = 0 -> y0 = 1<br>
>-----------------------------<br>
>result = 11 

>|       | Bit 1 | Bit 2 |
>| ----- |:-----:| -----:|
>|   X   |  0    | **1** |
>|   Y   | **1** |   0   |
>|   Z   | **1** | **1** |
>|   r   | **1** | **1** |

In [13]:
def maj(x, y, z):
    """
    :param x: 32-bit word
    :param y: 32-bit word
    :param z: 32-bit word

    :return: 32-bit word

    Majority function: for each bit position, the output bit is the majority value among the three input bits.
    """
    x = np.uint32(x); y = np.uint32(y); z = np.uint32(z)
    return np.uint32((x & y) | (x & z) | (y & z))

def majExamples():
    """
    Example cases demonstrating maj(x, y, z)
    """
    # Example cases
    examples = [
        ("all_zero",        np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0x00000000)),
        ("all_ones",        np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF)),
        ("two_ones_one_zero", np.uint32(0xFFFFFFFF), np.uint32(0xFFFFFFFF), np.uint32(0x00000000)),
        ("two_zeros_one_one", np.uint32(0x00000000), np.uint32(0x00000000), np.uint32(0xFFFFFFFF)),
        ("mixed_bits",      np.uint32(0xF0F0F0F0), np.uint32(0xAAAAAAAA), np.uint32(0x55555555)),
    ]

    print("Majority function examples.")
    # Test maj function against expected results
    for name, a, b, c in examples:
        res = np.uint32(maj(a, b, c))
        expected = np.uint32((a & b) | (a & c) | (b & c))
        assert res == expected, f"maj mismatch for {name}"
        print(f"{name}: x=0x{int(a):08x} y=0x{int(b):08x} z=0x{int(c):08x} -> maj=0x{int(res):08x} {int(res):032b}")
    print()

majExamples()

Majority function examples.
all_zero: x=0x00000000 y=0x00000000 z=0x00000000 -> maj=0x00000000 00000000000000000000000000000000
all_ones: x=0xffffffff y=0xffffffff z=0xffffffff -> maj=0xffffffff 11111111111111111111111111111111
two_ones_one_zero: x=0xffffffff y=0xffffffff z=0x00000000 -> maj=0xffffffff 11111111111111111111111111111111
two_zeros_one_one: x=0x00000000 y=0x00000000 z=0xffffffff -> maj=0x00000000 00000000000000000000000000000000
mixed_bits: x=0xf0f0f0f0 y=0xaaaaaaaa z=0x55555555 -> maj=0xf0f0f0f0 11110000111100001111000011110000



In [14]:
def Sigma0(x):
    x = np.uint32(x)
    res = (np.uint32(x >> 2) | np.uint32(x << np.uint32(32 - 2))) ^ (np.uint32(x >> 13) | np.uint32(x << np.uint32(32 - 13))) ^ (np.uint32(x >> 22) | np.uint32(x << np.uint32(32 - 22)))
    return np.uint32(res)

def Sigma1(x):
    x = np.uint32(x)
    res = (np.uint32(x >> 6) | np.uint32(x << np.uint32(32 - 6))) ^ (np.uint32(x >> 11) | np.uint32(x << np.uint32(32 - 11))) ^ (np.uint32(x >> 25) | np.uint32(x << np.uint32(32 - 25)))
    return np.uint32(res)

def sigma0(x):
    x = np.uint32(x)
    res = (np.uint32(x >> 7) | np.uint32(x << np.uint32(32 - 7))) ^ (np.uint32(x >> 18) | np.uint32(x << np.uint32(32 - 18))) ^ np.uint32(x >> 3)
    return np.uint32(res)

def sigma1(x):
    x = np.uint32(x)
    res = (np.uint32(x >> 17) | np.uint32(x << np.uint32(32 - 17))) ^ (np.uint32(x >> 19) | np.uint32(x << np.uint32(32 - 19))) ^ np.uint32(x >> 10)
    return np.uint32(res)

    




## Problem 2: Fractional Parts of Cube Roots

## Problem 3: Padding

## Problem 4: Hashes

## Problem 5: Passwords

## End